# Glovo Hourly Order Forecasting — Modeling

**Goal:** Forecast 168 hourly order counts for 2022-01-24 00:00:00 through 2022-01-30 23:00:00.

**Models compared:**
1. Naive baseline (lag-168)
2. OLS with time features
3. Prophet
4. LightGBM with lag features

**Validation:** Walk-forward cross-validation simulating the real production scenario (train up to Sunday, forecast the next full week).

**Metrics:** MSE and SMAPE.

## 0. Imports and config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '../data/train_data.csv'
TEST_MOCK_PATH = '../data/test_data_mock.csv'
FORECAST_START = pd.Timestamp('2022-01-24 00:00:00')
FORECAST_END   = pd.Timestamp('2022-01-30 23:00:00')
HORIZON = 168  # 7 days * 24 hours

## 1. Load and prepare data

In [ ]:
df = pd.read_csv(DATA_PATH)
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time').reset_index(drop=True)

# Fill the three known 6-hour gaps with 0 (overnight — structurally zero)
full_index = pd.date_range(df['time'].min(), df['time'].max(), freq='h')
df = df.set_index('time').reindex(full_index).rename_axis('time').reset_index()
df['orders'] = df['orders'].fillna(0.0)
df['city'] = df['city'].ffill()

print(f"Rows: {len(df)}")
print(f"Range: {df['time'].min()} to {df['time'].max()}")
df.head()

## 2. Feature engineering

In [ ]:
def add_time_features(df):
    df = df.copy()
    df['hour']        = df['time'].dt.hour
    df['dayofweek']   = df['time'].dt.dayofweek   # 0=Mon, 6=Sun
    df['week']        = df['time'].dt.isocalendar().week.astype(int)
    df['trend']       = np.arange(len(df))         # linear time index
    # Lag features (require sufficient history — NaN for first rows)
    df['lag_168']     = df['orders'].shift(168)    # same hour, last week
    df['lag_24']      = df['orders'].shift(24)     # same hour, yesterday
    df['roll_168_7d'] = df['orders'].shift(168).rolling(168).mean()  # avg same-week-slot over past week
    return df

df = add_time_features(df)
df.head()

## 3. Evaluation utilities

In [ ]:
def smape(actual, pred):
    actual, pred = np.array(actual), np.array(pred)
    denom = np.abs(actual) + np.abs(pred)
    return np.mean(np.where(denom == 0, 0, 2 * np.abs(actual - pred) / denom)) * 100

def mse(actual, pred):
    return np.mean((np.array(actual) - np.array(pred)) ** 2)

def evaluate(actual, pred, label=''):
    pred_clipped = np.maximum(pred, 0)
    m = mse(actual, pred_clipped)
    s = smape(actual, pred_clipped)
    if label:
        print(f"{label:30s}  MSE: {m:8.1f}  SMAPE: {s:.2f}%")
    return {'mse': m, 'smape': s}

## 4. Walk-forward cross-validation

Simulates the real production scenario: every Sunday we train on all available data and forecast the next full week (Mon 00:00 – Sun 23:00). We run this over the last N weeks of the training data to get out-of-sample scores.

In [ ]:
def get_cv_folds(df, n_folds=5):
    """
    Returns a list of (train_df, val_df) tuples.
    Each val_df is one full week (168 hours), the most recent n_folds weeks before the final cutoff.
    """
    folds = []
    # Last timestamp in data
    last = df['time'].max()
    # Work backwards: each fold's validation week ends at last - k*168 hours
    for k in range(n_folds, 0, -1):
        val_end   = last - pd.Timedelta(hours=(k - 1) * HORIZON)
        val_start = val_end - pd.Timedelta(hours=HORIZON - 1)
        train_end = val_start - pd.Timedelta(hours=1)

        train = df[df['time'] <= train_end].copy()
        val   = df[(df['time'] >= val_start) & (df['time'] <= val_end)].copy()

        if len(val) == HORIZON and len(train) >= HORIZON * 2:
            folds.append((train, val))

    return folds

folds = get_cv_folds(df, n_folds=5)
print(f"{len(folds)} folds created")
for i, (tr, va) in enumerate(folds):
    print(f"  Fold {i+1}: train ends {tr['time'].max()}  |  val: {va['time'].min()} to {va['time'].max()}")

## 5. Model 1 — Naive baseline (lag-168)

Predict each hour as the same hour from the previous week. No fitting required.

In [ ]:
def predict_naive(train, val):
    # Last 168 hours of training data = previous week
    last_week = train.tail(HORIZON)['orders'].values
    return np.maximum(last_week, 0)

naive_scores = []
for train, val in folds:
    preds = predict_naive(train, val)
    naive_scores.append(evaluate(val['orders'].values, preds))

naive_avg = {k: np.mean([s[k] for s in naive_scores]) for k in ['mse', 'smape']}
print(f"Naive baseline   avg MSE: {naive_avg['mse']:.1f}   avg SMAPE: {naive_avg['smape']:.2f}%")

## 6. Model 2 — OLS with time features

Linear regression with hour-of-day dummies, day-of-week dummies, and a linear trend term.

In [ ]:
from sklearn.linear_model import LinearRegression

OLS_FEATURES = ['trend', 'hour', 'dayofweek']

def predict_ols(train, val):
    X_train = pd.get_dummies(train[OLS_FEATURES], columns=['hour', 'dayofweek'], drop_first=True)
    X_val   = pd.get_dummies(val[OLS_FEATURES],   columns=['hour', 'dayofweek'], drop_first=True)
    # Align columns (val may be missing some dummies if a category didn't appear)
    X_val = X_val.reindex(columns=X_train.columns, fill_value=0)

    model = LinearRegression()
    model.fit(X_train, train['orders'])
    return np.maximum(model.predict(X_val), 0)

ols_scores = []
for train, val in folds:
    preds = predict_ols(train, val)
    ols_scores.append(evaluate(val['orders'].values, preds))

ols_avg = {k: np.mean([s[k] for s in ols_scores]) for k in ['mse', 'smape']}
print(f"OLS              avg MSE: {ols_avg['mse']:.1f}   avg SMAPE: {ols_avg['smape']:.2f}%")

## 7. Model 3 — Prophet

Handles daily + weekly seasonality and trend natively. No yearly seasonality (less than 2 full cycles in data).

In [ ]:
from prophet import Prophet

def predict_prophet(train, val):
    prophet_df = train[['time', 'orders']].rename(columns={'time': 'ds', 'orders': 'y'})
    m = Prophet(
        yearly_seasonality=False,
        weekly_seasonality=True,
        daily_seasonality=True,
        seasonality_mode='additive',
    )
    m.fit(prophet_df)
    future = val[['time']].rename(columns={'time': 'ds'})
    forecast = m.predict(future)
    return np.maximum(forecast['yhat'].values, 0)

prophet_scores = []
for train, val in folds:
    preds = predict_prophet(train, val)
    prophet_scores.append(evaluate(val['orders'].values, preds))

prophet_avg = {k: np.mean([s[k] for s in prophet_scores]) for k in ['mse', 'smape']}
print(f"Prophet          avg MSE: {prophet_avg['mse']:.1f}   avg SMAPE: {prophet_avg['smape']:.2f}%")

## 8. Model 4 — LightGBM with lag features

Tree-based model with engineered lag and time features. Strong for multi-seasonal data.

**Note:** lag features are computed from training data only — no leakage into validation.

In [ ]:
import lightgbm as lgb

LGBM_FEATURES = ['hour', 'dayofweek', 'trend', 'lag_168', 'lag_24', 'roll_168_7d']

def predict_lgbm(train, val):
    tr = train.dropna(subset=LGBM_FEATURES)
    X_train = tr[LGBM_FEATURES]
    y_train = tr['orders']

    # For val, lag features must come from the tail of training data (no leakage)
    combined = pd.concat([train, val]).reset_index(drop=True)
    combined = add_time_features(combined)
    X_val = combined.loc[combined['time'].isin(val['time']), LGBM_FEATURES]

    model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=31, random_state=42, verbose=-1)
    model.fit(X_train, y_train)
    return np.maximum(model.predict(X_val), 0)

lgbm_scores = []
for train, val in folds:
    preds = predict_lgbm(train, val)
    lgbm_scores.append(evaluate(val['orders'].values, preds))

lgbm_avg = {k: np.mean([s[k] for s in lgbm_scores]) for k in ['mse', 'smape']}
print(f"LightGBM         avg MSE: {lgbm_avg['mse']:.1f}   avg SMAPE: {lgbm_avg['smape']:.2f}%")

## 9. Results comparison

In [ ]:
results = pd.DataFrame([
    {'model': 'Naive (lag-168)', **naive_avg},
    {'model': 'OLS',             **ols_avg},
    {'model': 'Prophet',         **prophet_avg},
    {'model': 'LightGBM',        **lgbm_avg},
]).set_index('model').round(2)

results.columns = ['Avg MSE', 'Avg SMAPE (%)']
print(results.to_string())

# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
results['Avg MSE'].plot(kind='bar', ax=axes[0], title='Avg MSE (lower is better)', color='steelblue')
results['Avg SMAPE (%)'].plot(kind='bar', ax=axes[1], title='Avg SMAPE % (lower is better)', color='coral')
for ax in axes:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 10. Champion model — refit and generate final forecast

Select the best model from Section 9, refit on all available training data, and forecast Jan 24–30 2022.

In [ ]:
# TODO: swap in whichever function performed best above
CHAMPION = 'LightGBM'  # update after seeing results

forecast_index = pd.date_range(FORECAST_START, FORECAST_END, freq='h')
forecast_df = pd.DataFrame({'time': forecast_index})

# Build a combined df for lag computation
full = pd.concat([df, forecast_df]).reset_index(drop=True)
full = add_time_features(full)

val_rows = full[full['time'].isin(forecast_index)]

# --- Fit champion on all training data and predict ---
# Replace this block with the chosen model's predict function
X_all   = df.dropna(subset=LGBM_FEATURES)[LGBM_FEATURES]
y_all   = df.dropna(subset=LGBM_FEATURES)['orders']
X_final = val_rows[LGBM_FEATURES]

champion_model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=31, random_state=42, verbose=-1)
champion_model.fit(X_all, y_all)
final_preds = np.maximum(champion_model.predict(X_final), 0)

## 11. Build predictions DataFrame and format check

In [ ]:
import sys
sys.path.append('..')
from check_output_format import check_output_format

predictions = pd.DataFrame({
    'time': forecast_index,
    'preds': final_preds.astype('float64'),
})

print(f"Shape: {predictions.shape}")
print(f"Dtypes:\n{predictions.dtypes}")
print(f"Nulls: {predictions.isnull().sum().sum()}")
predictions.head()

In [ ]:
# Run the official format checker
check_output_format(predictions, TEST_MOCK_PATH)

## 12. Save and verify

In [ ]:
OUTPUT_PATH = '../data/predictions.csv'
predictions.to_csv(OUTPUT_PATH, index=False)

# Re-read and verify dtypes and row count
check_df = pd.read_csv(OUTPUT_PATH, parse_dates=['time'])
assert len(check_df) == 168, f"Expected 168 rows, got {len(check_df)}"
assert check_df['preds'].dtype == 'float64', "preds must be float64"
assert check_df['time'].dtype == 'datetime64[ns]', "time must be datetime64[ns]"
assert check_df.isnull().sum().sum() == 0, "Found null values"
print(f"Saved to {OUTPUT_PATH} — all checks passed.")